## Data Collection Pipeline

In [ ]:
import pandas as pd
import json
import os
from pathlib import Path
import ast

DATASET_PATH = Path("../../data")

In [ ]:
### Loading data into pandas dataframe

messages = []

for group_title in os.listdir(DATASET_PATH / "raw"):
    file_path = DATASET_PATH / "raw" / group_title

    with open(file_path, "r", encoding="utf-8") as group_file:
        for lidx, line in enumerate(group_file):
            message = json.loads(line)

            from_id = message.get("from_id") or {}
            peer_id = message.get("peer_id") or {}
            reply_to = message.get("reply_to") or {}
            
            messages.append({
                "group_title": group_title,
                "message_id": message.get("id"),
                "user_id": from_id.get("user_id"),
                "channel_id": peer_id.get("channel_id"),
                "date": message.get("date"),
                "text": message.get("message").strip() or "",
                "reply_to": (reply_to or {}).get("reply_to_msg_id")
            })

df = pd.DataFrame(messages)


In [ ]:
## See dataset info
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 198058 entries, 0 to 198057
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   group_title  198058 non-null  str    
 1   message_id   198058 non-null  int64  
 2   user_id      196337 non-null  float64
 3   channel_id   198058 non-null  int64  
 4   date         198058 non-null  str    
 5   text         198058 non-null  str    
 6   reply_to     58206 non-null   float64
 7   reply_text   49955 non-null   str    
dtypes: float64(2), int64(2), str(4)
memory usage: 12.1 MB


In [95]:
## Clean text column by stripping new lines

df["text"] = (
    df["text"]
    .fillna("")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [96]:
df = df[df["text"].str.strip().ne("")]

In [97]:
df["reply_text"] = (
    df[["channel_id", "reply_to"]]
    .merge(
        df[["channel_id", "message_id", "text"]],
        left_on=["channel_id", "reply_to"],
        right_on=["channel_id", "message_id"],
        how="left"
    )["text"]
    .to_numpy()
)

In [91]:
df.head()

,group_title,message_id,user_id,channel_id,date,text,reply_to,reply_text
1,group_-1001548974049.jsonl,85533,5.538031e+09,1548974049,2026-09-24 14:01:04+00:00,🥳🤬☹️🥳😖🥳😠😠🥳 🇺🇿🇺🇿 Yo‘nalishlar: ✅Toshkent ➡️ Inc...,NaN,NaN
2,group_-1001548974049.jsonl,85532,6.376728e+09,1548974049,2026-09-24 12:42:24+00:00,Assalomu alaykum yaxshimisizlar iphone 16 pro ...,NaN,NaN
3,group_-1001548974049.jsonl,85531,8.894030e+09,1548974049,2026-09-24 11:43:15+00:00,"Assalomu alaykum, hurmatli yurtdoshlar. Yaxshi...",NaN,NaN
4,group_-1001548974049.jsonl,85530,8.606855e+09,1548974049,2026-09-24 11:40:51+00:00,Akalar gruppani odamlarga qulaylik bolsin deb ...,NaN,NaN
5,group_-1001548974049.jsonl,85529,1.958333e+08,1548974049,2026-09-24 11:30:29+00:00,Oka bunday qilib qaydagi qizni deb xurmat g‘ur...,NaN,NaN


In [92]:
df.info()

<class 'pandas.DataFrame'>
Index: 163130 entries, 1 to 198056
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   group_title  163130 non-null  str    
 1   message_id   163130 non-null  int64  
 2   user_id      161711 non-null  float64
 3   channel_id   163130 non-null  int64  
 4   date         163130 non-null  str    
 5   text         163130 non-null  str    
 6   reply_to     56154 non-null   float64
 7   reply_text   43891 non-null   str    
dtypes: float64(2), int64(2), str(4)
memory usage: 11.2 MB


In [93]:
len(df.user_id.unique())

14829

In [94]:
df.to_csv(DATASET_PATH / "telegram_messages.csv", index=False, encoding="utf-8-sig")